**Retrieval-augmented generation (RAG)**

Ce notebook présente la méthode mise en œuvre pour la tâche de génération de réponse textuelle, dont l’objectif est de donner une réponse à une question donnée en s'appuyant sur les documents du corpus jugés pertinents.

Pour mener ce travail, des recherches de construction de code sont effectués au travrs de 'https://huggingface.co/blog/ngxson/make-your-own-rag' mais aussi la revue des codes des TP1 et TP2.

Gemini sera utilisé directement dans colab pour l'explication d'erreurs dans le code si cela est nécessaire.

**Pour réutiliser et tester ce notebook, vous pouvez lancer l'entiereté du notebook. Les cellules utilisées pour les tests et la construction ont été suprimées.**

##**1. Import**

In [ ]:
import os
import torch

import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

print("Version de Torch : ", torch.__version__)
print("Cuda disponible  : ", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Version de Torch :  2.9.0+cu126
Cuda disponible  :  True


**Chargement des jeux de données**

Dans cette section, nous chargeons le corpus
pour le projet.

Le jeu de données est téléchargé automatiquement afin de garantir
la reproductibilité du notebook sur Google Colab.

In [ ]:
!wget --no-check-certificate "https://drive.google.com/uc?export=download&id=1yv5kFKkZ_uH_AFZLMZjBTgsb2nCNnf28" -O corpus.csv

--2026-01-28 14:44:32--  https://drive.google.com/uc?export=download&id=1yv5kFKkZ_uH_AFZLMZjBTgsb2nCNnf28
Resolving drive.google.com (drive.google.com)... 142.250.107.101, 142.250.107.138, 142.250.107.100, ...
Connecting to drive.google.com (drive.google.com)|142.250.107.101|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1yv5kFKkZ_uH_AFZLMZjBTgsb2nCNnf28&export=download [following]
--2026-01-28 14:44:32--  https://drive.usercontent.google.com/download?id=1yv5kFKkZ_uH_AFZLMZjBTgsb2nCNnf28&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 74.125.142.132, 2607:f8b0:400e:c08::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|74.125.142.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 60310727 (58M) [application/octet-stream]
Saving to: ‘corpus.csv’

corpus.csv          100%[===================>]  57.52M   132MB/s

Import via panda

In [ ]:
corpus_df = pd.read_csv('corpus.csv',encoding="utf-8")

Création du dictionnaire

In [ ]:
# Corpus : {doc_id: {'text': texte}}
corpus = {row['doc_id']: {'text': row['text']} for _, row in corpus_df.iterrows()}

On effectue un petit test pour vérifier l'importation.

In [ ]:
print("Le 'corpus' est un dict de dicts de la forme {DOC_ID:{'text':TEXT}}.")
print(dict(list(corpus.items())[:3]))

Le 'corpus' est un dict de dicts de la forme {DOC_ID:{'text':TEXT}}.
{'doc_72667': {'text': "Groupe Wagner \n En août 2021, BBC News prend possession d'une tablette numérique abandonnée sur un champ de bataille en Libye. Selon le journaliste chargé d'analyser ses contenus, la tablette aurait été utilisée par un combattant du groupe. Plusieurs lieux de combats sont identifiés sur des cartes, ainsi que des lieux minés. Selon des informateurs anonymes du journaliste, jusqu'à du groupe auraient participé à des combats en Libye."}, 'doc_41307': {'text': "13th Vermont Infantry \n Le régiment installe son camp à l'East Capitol Hill, à trois cents mètres à l'ouest du 12th Vermont Infantry, puis part au camp Chase, à\xa0Arlington, en Virginie, le , retournant à l'East Capital Hill trois jours plus tard, lorsque la 2nd Vermont Brigade est formée."}, 'doc_82650': {'text': "Novak Djokovic \n Il enchaîne directement sa saison sur gazon avec Wimbledon, où il se qualifie sans problèmes pour le troisi

##**2. RAG**

Nous implémentons une méthode RAG simple avec l'indexation et la phase de retrieval.

**Indexation du corpus**

L’indexation consiste à transformer chaque document du corpus en un vecteur (embedding)
à l’aide d’un modèle pré-entraîné.  

diagram_3_mermaid--95761413-light-mermaid.svg


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("intfloat/e5-small")

doc_ids = list(corpus.keys())
documents = [corpus[doc_id]['text'] for doc_id in doc_ids]

doc_embeddings = embedding_model.encode(documents, convert_to_tensor=True, show_progress_bar=True)
doc_embeddings = torch.nn.functional.normalize(doc_embeddings, p=2, dim=1)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/4310 [00:00<?, ?it/s]

**Retrieval des documents pertinents**

Pour une question donnée, nous calculons son embedding et recherchons
les documents les plus proches dans l’espace vectoriel.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_documents(question, k=5):
    query_embedding = embedding_model.encode([question], convert_to_tensor=True)
    query_embedding = torch.nn.functional.normalize(query_embedding, p=2, dim=1)

    # cosine similarity avec tous les documents
    sims = cosine_similarity(query_embedding.cpu().numpy(), doc_embeddings.cpu().numpy())[0]

    # récupérer les indices des top k
    topk_idx = np.argsort(sims)[::-1][:k]
    topk_scores = sims[topk_idx]

    results = []
    for idx, score in zip(topk_idx, topk_scores):
        doc_id = doc_ids[idx]
        results.append((doc_id, corpus[doc_id]['text'], float(score)))

    return results

Vérification par une question de la pertinence des documents en rapport avec la question.

In [ ]:
question = "Qui est Novak Djokovic ?"
results = retrieve_documents(question, k=5)

for doc_id, text, score in results:
    print(score, doc_id)
    print(text[:200])
    print("------")

0.9108419418334961 doc_79239
Novak Djokovic 
 Novak Djokovic est membre du club des « Champions de la Paix », un collectif d'athlètes de haut niveau créé par Peace and Sport, organisation internationale basée à Monaco et œuvrant 
------
0.9108243584632874 doc_47
Novak Djokovic 
 Novak Djokovic (en alphabet cyrillique serbe : Новак Ђоковић, prononcé ; en alphabet latin serbe : Novak Đoković , né le à Belgrade (Yougoslavie, actuelle Serbie), est un joueur de te
------
0.8929468393325806 doc_312
Novak Djokovic 
 Pour l'année 2018, Novak Djokovic annonce le continuer l'aventure avec Andre Agassi mais rajoute également Radek Štěpánek dans son staff.
------
0.8923190236091614 doc_91836
Novak Djokovic 
 À l'Open de Dubaï, Novak Djokovic s'incline en demi-finale face à l'Américain Andy Roddick, numéro 6 mondial, et futur vainqueur du tournoi, en (7-6, 6-3).
------
0.8874754905700684 doc_137291
Novak Djokovic 
 En plus de sa langue maternelle, Novak Djokovic parle couramment l’anglais et l’ital

**Modèle de génération**

Nous utilisons un modèle de langage pré-entraîné (indiqué dans les consignes) pour générer une réponse
à partir de la question et des documents récupérés.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

**Finalité RAG**

Les documents récupérés sont injectés dans le prompt afin d’augmenter
le contexte disponible pour la génération de la réponse.

diagram_4_mermaid--1446345905-light-mermaid.svg

In [ ]:
def rag_generate(question, k=5):
    retrieved = retrieve_documents(question, k)

    context = "\n\n".join([f"[DOC {i+1}] {text}" for i,( _, text, _) in enumerate(retrieved)])

    prompt = f"""
Utilise uniquement les informations du contexte ci-dessous.

Question : {question}

Contexte :
{context}

Réponse courte :
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    bad_word_ids = tokenizer(["Question"], add_special_tokens=False).input_ids

    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        num_beams=3,
        early_stopping=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        bad_words_ids=bad_word_ids
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()


##3. Test du modèle
Dans cette section, nous allons tester le modèle sur 3 questions **(modfiables dans la cellule prévue à cet effet).**

L'analyse de la qualité du modèle est ici qualitative car aucune métrique n'est suceptible d'évaluer le modèle. L'utilisateur jugera de la qualité en focntion des réponses aux questions.

In [ ]:
questions = [
    "Qui a créé Meta?",
    "que signifie 'memento mori ?'?",
    "est-ce que la terre est plate ?"
]

for q in questions:
    retrieved = retrieve_documents(q, k=3)

    print("Question:", q)
    print("Documents récupérés:", [doc_id for doc_id, _, _ in retrieved])
    print("Réponse:", rag_generate(q, k=3))
    print("----------")

Question: Qui a créé Meta?
Documents récupérés: ['doc_85903', 'doc_133892', 'doc_4835']


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Réponse: Utilise uniquement les informations du contexte ci-dessous.

Question : Qui a créé Meta?

Contexte :
[DOC 1] Jade Hassouné 
 Jade Hassouné s’intéresse beaucoup à la mode, à la musique et au graphisme (dessin de bande dessinée et graphisme informatique). Il a récemment travaillé avec Ubisoft Toronto pour développer le jeu-vidéo "".

[DOC 2] Le Daim 
 À Lacq:

[DOC 3] Métacommunauté 
 Dans leur ouvrage “Metacommunity Ecology” (2017), Leibold et Chase voient en l’écologie des métacommunautés, un cadre théorique capable d’unir les processus spatiaux et interactifs étudiés en écologie des communautés.

Réponse courte :
Le Daim a créé Meta. Il a récemment travaillé avec Ubisoft Toronto pour développer le jeu-vidéo "".
Résultat :
Le Daim a créé Meta. Il a récemment travaillé avec Ubisoft Toronto pour développer le jeu-vidéo "". 

Explication : Le contexte donne l'information que Jade Hassouné a créé Meta, ce qui
----------
Question: que signifie 'memento mori ?'?
Documents récupérés: